In [1]:
import pandas as pd
from qdrant_client import QdrantClient, models

## Reading the DataFrame

In [2]:
df = pd.read_csv('data/data.csv')

In [3]:
df.head()

,food_name,food_type,serving_size,meal_type,calories_kcal,protein_g,carbs_g,fats_g,saturated_fat_g,monounsaturated_fat_g,polyunsaturated_fat_g,trans_fat_g,fiber_g,sodium_mg,calcium_mg,phosphorus_mg
0,Oats Porridge,veg,100g cooked,breakfast,210,7.40,44.24,12.26,5.92,4.11,3.15,0.17,1.22,101,235,200
1,Idli,veg,100g cooked,breakfast,233,8.23,12.96,9.65,3.42,2.70,1.55,0.04,4.44,298,113,337
2,Dosa,veg,100g cooked,breakfast,245,8.97,12.34,6.66,1.71,1.70,1.37,0.03,4.59,115,267,77
3,Upma,veg,100g cooked,breakfast,293,20.20,12.86,14.78,6.93,4.56,3.81,0.03,6.79,203,96,221
4,Poha,veg,100g cooked,breakfast,306,24.12,26.64,6.28,1.79,1.97,1.54,0.12,1.80,551,203,76


In [4]:
documents = df.to_dict(orient='records')

In [5]:
documents[0]

{'food_name': 'Oats Porridge',
 'food_type': 'veg',
 'serving_size': '100g cooked',
 'meal_type': 'breakfast',
 'calories_kcal': 210,
 'protein_g': 7.4,
 'carbs_g': 44.24,
 'fats_g': 12.26,
 'saturated_fat_g': 5.92,
 'monounsaturated_fat_g': 4.11,
 'polyunsaturated_fat_g': 3.15,
 'trans_fat_g': 0.17,
 'fiber_g': 1.22,
 'sodium_mg': 101,
 'calcium_mg': 235,
 'phosphorus_mg': 200}

# Retrieval

## Loading Qdrant

In [6]:
client = QdrantClient("http://localhost:6333")

In [7]:
from fastembed import TextEmbedding
TextEmbedding.list_supported_models()

[{'model': 'BAAI/bge-base-en',
  'sources': {'hf': 'Qdrant/fast-bge-base-en',
   'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en.tar.gz',
   '_deprecated_tar_struct': True},
  'model_file': 'model_optimized.onnx',
  'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: necessary, 2023 year.',
  'license': 'mit',
  'size_in_GB': 0.42,
  'additional_files': [],
  'dim': 768,
  'tasks': {}},
 {'model': 'BAAI/bge-base-en-v1.5',
  'sources': {'hf': 'qdrant/bge-base-en-v1.5-onnx-q',
   'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en-v1.5.tar.gz',
   '_deprecated_tar_struct': True},
  'model_file': 'model_optimized.onnx',
  'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.',
  'license': 'mit',
  'size_in_GB': 0.21,
  'additional_files': [],
  'dim': 768,
  'tasks': {}},
 {'model':

### Choosing correct Model

In [8]:
import json

EMBEDDING_DIMENSIONALITY = 384

for model in TextEmbedding.list_supported_models():
    if model["dim"] == EMBEDDING_DIMENSIONALITY:
        print(json.dumps(model, indent=2))

{
  "model": "BAAI/bge-small-en",
  "sources": {
    "hf": "Qdrant/bge-small-en",
    "url": "https://storage.googleapis.com/qdrant-fastembed/BAAI-bge-small-en.tar.gz",
    "_deprecated_tar_struct": true
  },
  "model_file": "model_optimized.onnx",
  "description": "Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: necessary, 2023 year.",
  "license": "mit",
  "size_in_GB": 0.13,
  "additional_files": [],
  "dim": 384,
  "tasks": {}
}
{
  "model": "BAAI/bge-small-en-v1.5",
  "sources": {
    "hf": "qdrant/bge-small-en-v1.5-onnx-q",
    "url": null,
    "_deprecated_tar_struct": false
  },
  "model_file": "model_optimized.onnx",
  "description": "Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.",
  "license": "mit",
  "size_in_GB": 0.067,
  "additional_files": [],
  "dim": 384,
  "tasks": {}
}
{
  "model": "snowflake/snowflake-arctic-embed-xs",
  "sou

In [9]:
model_handle =  "BAAI/bge-small-en-v1.5"
collection_name = 'NutriRAG'

In [10]:
client.recreate_collection(                     #If collection already exists
    collection_name = collection_name,
    vectors_config = models.VectorParams(
        size = 384,
        distance=models.Distance.COSINE
    )
)


/tmp/ipykernel_5174/49177396.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(                     #If collection already exists


True

In [11]:
points = []
id = 0

In [12]:
for doc in documents:
        point = models.PointStruct(
            id=id,
            vector=models.Document(text = doc['food_name'], model = model_handle),
            payload = {
                "food_name": doc["food_name"],
                "meal_type": doc["meal_type"],
                "food_type": doc["food_type"],
                "serving_size": doc["serving_size"],

                # Macros
                "calories_kcal": doc["calories_kcal"],
                "protein_g": doc["protein_g"],
                "carbs_g": doc["carbs_g"],
                "fats_g": doc["fats_g"],
                "fiber_g": doc["fiber_g"],

                # Fat breakdown
                "saturated_fat_g": doc["saturated_fat_g"],
                "monounsaturated_fat_g": doc["monounsaturated_fat_g"],
                "polyunsaturated_fat_g": doc["polyunsaturated_fat_g"],
                "trans_fat_g": doc["trans_fat_g"],

                # Micros
                "sodium_mg": doc["sodium_mg"],
                "calcium_mg": doc["calcium_mg"],
                "phosphorus_mg": doc["phosphorus_mg"]
            }
        )

        points.append(point)
        id+=1

In [13]:
client.upsert(
    collection_name=collection_name,
    points=points
    )

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [14]:
def vector_search(query, limit = 5):
    results = client.query_points(
        collection_name = collection_name,
        query = models.Document(
            text=query,
            model = model_handle
        ),
    limit = limit,
    with_payload = True
    )
    return results
    

In [15]:
test = vector_search('High protein non-vegetarian dinner')
test

QueryResponse(points=[ScoredPoint(id=8, version=1, score=0.76466244, payload={'food_name': 'Protein Oats', 'meal_type': 'breakfast', 'food_type': 'veg', 'serving_size': '100g cooked', 'calories_kcal': 239, 'protein_g': 19.26, 'carbs_g': 24.79, 'fats_g': 8.8, 'fiber_g': 4.55, 'saturated_fat_g': 2.23, 'monounsaturated_fat_g': 3.0, 'polyunsaturated_fat_g': 2.3, 'trans_fat_g': 0.01, 'sodium_mg': 287, 'calcium_mg': 250, 'phosphorus_mg': 20}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=91, version=1, score=0.7450124, payload={'food_name': 'Protein Bar', 'meal_type': 'snacks', 'food_type': 'veg', 'serving_size': '1 bar', 'calories_kcal': 336, 'protein_g': 1.53, 'carbs_g': 15.76, 'fats_g': 16.01, 'fiber_g': 4.69, 'saturated_fat_g': 3.5, 'monounsaturated_fat_g': 6.36, 'polyunsaturated_fat_g': 1.68, 'trans_fat_g': 0.31, 'sodium_mg': 348, 'calcium_mg': 134, 'phosphorus_mg': 219}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=14, version=1, score=0.73691744, pay

In [16]:
test.points

[ScoredPoint(id=8, version=1, score=0.76466244, payload={'food_name': 'Protein Oats', 'meal_type': 'breakfast', 'food_type': 'veg', 'serving_size': '100g cooked', 'calories_kcal': 239, 'protein_g': 19.26, 'carbs_g': 24.79, 'fats_g': 8.8, 'fiber_g': 4.55, 'saturated_fat_g': 2.23, 'monounsaturated_fat_g': 3.0, 'polyunsaturated_fat_g': 2.3, 'trans_fat_g': 0.01, 'sodium_mg': 287, 'calcium_mg': 250, 'phosphorus_mg': 20}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=91, version=1, score=0.7450124, payload={'food_name': 'Protein Bar', 'meal_type': 'snacks', 'food_type': 'veg', 'serving_size': '1 bar', 'calories_kcal': 336, 'protein_g': 1.53, 'carbs_g': 15.76, 'fats_g': 16.01, 'fiber_g': 4.69, 'saturated_fat_g': 3.5, 'monounsaturated_fat_g': 6.36, 'polyunsaturated_fat_g': 1.68, 'trans_fat_g': 0.31, 'sodium_mg': 348, 'calcium_mg': 134, 'phosphorus_mg': 219}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=14, version=1, score=0.73691744, payload={'food_name': 

In [17]:
print(test)

points=[ScoredPoint(id=8, version=1, score=0.76466244, payload={'food_name': 'Protein Oats', 'meal_type': 'breakfast', 'food_type': 'veg', 'serving_size': '100g cooked', 'calories_kcal': 239, 'protein_g': 19.26, 'carbs_g': 24.79, 'fats_g': 8.8, 'fiber_g': 4.55, 'saturated_fat_g': 2.23, 'monounsaturated_fat_g': 3.0, 'polyunsaturated_fat_g': 2.3, 'trans_fat_g': 0.01, 'sodium_mg': 287, 'calcium_mg': 250, 'phosphorus_mg': 20}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=91, version=1, score=0.7450124, payload={'food_name': 'Protein Bar', 'meal_type': 'snacks', 'food_type': 'veg', 'serving_size': '1 bar', 'calories_kcal': 336, 'protein_g': 1.53, 'carbs_g': 15.76, 'fats_g': 16.01, 'fiber_g': 4.69, 'saturated_fat_g': 3.5, 'monounsaturated_fat_g': 6.36, 'polyunsaturated_fat_g': 1.68, 'trans_fat_g': 0.31, 'sodium_mg': 348, 'calcium_mg': 134, 'phosphorus_mg': 219}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=14, version=1, score=0.73691744, payload={'food_na

In [18]:
for t in test.points:
    print(t.payload['food_name'], t.payload['meal_type'], t.payload['food_type'], t.payload['protein_g'])

Protein Oats breakfast veg 19.26
Protein Bar snacks veg 1.53
Protein Shake breakfast veg 21.19
Veg Sandwich breakfast veg 1.98
Vegetable Sandwich snacks veg 15.08


Why nutrition values are NOT stored in embeddings

Embedding models (BGE-small, MiniLM, OpenAI, etc.) capture semantic meaning, not numeric magnitude.
Numeric nutrition values like:

protein_g

calories_kcal

sodium_mg

cannot be understood correctly by embeddings.
For example:

“Protein Bar” may embed strongly for “protein”
even if the actual protein value is low.

To solve this, the search engine uses a two-stage ranking pipeline:

Semantic Recall (embedding-based)
Retrieve foods that are semantically related to the query.

Nutrition-Aware Re-Ranking (metadata-based)
Sort or score results using actual nutrition values
(e.g., highest protein, lowest calories, lowest sodium).

This produces search results that are both semantically relevant and nutritionally correct, which is essential for diet planning.

In [19]:
client.create_payload_index(
    collection_name = collection_name,
    field_name = 'meal_type',
    field_schema = 'keyword'
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [20]:
client.create_payload_index(
    collection_name = collection_name,
    field_name = 'food_type',
    field_schema = 'keyword'
)

UpdateResult(operation_id=5, status=<UpdateStatus.COMPLETED: 'completed'>)

In [21]:
query = 'Suggest me high protein non veg dinner'

In [22]:
def extract_query(query):
    query = query.lower()

    meal_type = None
    food_type = None
    rank_by = None

    # ---- Food type filters ----
    veg_terms = {"veg", "vegetarian", "plant based"}
    non_veg_terms = {"non veg", "chicken", "fish", "egg", "non-veg"}

    if any(term in query for term in non_veg_terms):
        food_type = "non-veg"
    elif any(term in query for term in veg_terms):
        food_type = "veg"

    # ---- Meal type filters ----
    if "breakfast" in query:
        meal_type = "breakfast"
    elif "lunch" in query:
        meal_type = "lunch"
    elif "dinner" in query:
        meal_type = "dinner"
    elif "snack" in query:
        meal_type = "snacks"

    # ---- Ranking intent (NOT filters) ----
    if "high protein" in query:
        rank_by = ("protein_g", True)
    elif "high carb" in query:
        rank_by = ("carbs_g", True)
    elif "low calorie" in query:
        rank_by = ("calories_kcal", False)

    return food_type, meal_type, rank_by


In [23]:
def search_with_filter(query, meal_type='', food_type = '', limit = 20):

    must_conditions = []

    if meal_type:
        must_conditions.append(
            models.FieldCondition(
                    key= 'meal_type',
                    match=models.MatchValue(value=meal_type)

            )
        )
    if food_type:
        must_conditions.append(
            models.FieldCondition(
                    key= 'food_type',
                    match=models.MatchValue(value=food_type)
                )
        )

    
        
    results = client.query_points(
        collection_name = collection_name,
        query = models.Document(
            text=query,
            model = model_handle
        ),
    
        query_filter=models.Filter( 
            must=must_conditions if must_conditions else None
        ),
        limit=limit,
        with_payload=True 
    )

    return results


In [50]:
def search(query, limit=10):
    food, meal, rank = extract_query(query)
    result = search_with_filter(query = query,meal_type = meal,food_type = food, limit = limit * 5)
    points = result.points
    if rank:
        field, descend = rank
        points = sorted(points, key = lambda p:p.payload[field],
        reverse = descend
                       )
    return points[:limit]

In [25]:
test = search(query, 10)
test

[ScoredPoint(id=67, version=1, score=0.6657377, payload={'food_name': 'Chicken Clear Soup', 'meal_type': 'dinner', 'food_type': 'non-veg', 'serving_size': '100g cooked', 'calories_kcal': 186, 'protein_g': 25.09, 'carbs_g': 14.78, 'fats_g': 17.36, 'fiber_g': 1.9, 'saturated_fat_g': 6.23, 'monounsaturated_fat_g': 4.78, 'polyunsaturated_fat_g': 4.57, 'trans_fat_g': 0.33, 'sodium_mg': 237, 'calcium_mg': 20, 'phosphorus_mg': 182}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=59, version=1, score=0.64094305, payload={'food_name': 'Egg Curry', 'meal_type': 'dinner', 'food_type': 'non-veg', 'serving_size': '100g cooked', 'calories_kcal': 130, 'protein_g': 24.16, 'carbs_g': 13.0, 'fats_g': 3.27, 'fiber_g': 0.51, 'saturated_fat_g': 1.02, 'monounsaturated_fat_g': 0.94, 'polyunsaturated_fat_g': 0.55, 'trans_fat_g': 0.05, 'sodium_mg': 161, 'calcium_mg': 166, 'phosphorus_mg': 69}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=48, version=1, score=0.6415145, paylo

In [26]:
for t in test:
    print(t.payload['food_name'], t.payload['meal_type'], t.payload['food_type'], t.payload['protein_g'])

Chicken Clear Soup dinner non-veg 25.09
Egg Curry dinner non-veg 24.16
Grilled Chicken Breast dinner non-veg 16.64
Baked Fish dinner non-veg 11.43
Chicken Soup dinner non-veg 5.77


# Generation

In [27]:
import google.generativeai as genai
genai.configure()
model = genai.GenerativeModel("gemini-2.5-flash")

/tmp/ipykernel_5174/1861333358.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [28]:
response = model.generate_content(query)

In [29]:
print(response.text)

Here are some high-protein non-veg dinner suggestions that are delicious, satisfying, and easy to prepare:

**Key Principles for High Protein Dinner:**
*   **Focus on Lean Protein:** Chicken breast, turkey, fish (salmon, cod, tuna), lean cuts of beef (sirloin, flank), pork tenderloin, shrimp.
*   **Generous Protein Portion:** Aim for 4-6 oz (110-170g) cooked protein per serving.
*   **Pair with Vegetables:** Load up on non-starchy vegetables for fiber, vitamins, and volume without adding excessive calories.
*   **Healthy Fats (in moderation):** Olive oil, avocado, nuts/seeds, fatty fish.
*   **Complex Carbs (optional, in moderation):** Quinoa, brown rice, sweet potato, whole grains.

---

### High Protein Non-Veg Dinner Suggestions:

1.  **Lemon Herb Grilled/Baked Chicken Breast with Roasted Asparagus & Quinoa**
    *   **Protein:** Chicken breast (marinade with lemon juice, olive oil, garlic, dried herbs like oregano, thyme, rosemary).
    *   **Preparation:** Grill, pan-sear, or bake

In [30]:
## Building a prompt

prompt_template = """
You are a professional dietician and nutrition expert.

Answer the QUESTION using ONLY the food items and nutrition information listed below.
Do NOT invent foods, nutrition values, or meals that are not provided.

Your responsibilities:
- Select suitable food items based on the user's request
- Combine them into a sensible meal or meal plan
- Match the requested meal type, food preference, and nutrition goal
- Use a clear, professional, and natural tone.
- Do not mention any reference data in output

IMPORTANT OUTPUT RULES:
1. Always list each selected food item separately.
2. For EACH food item, include:
   - Food name
   - Serving size
   - Calories
   - Protein
   - Carbohydrates
   - Fats
3. After listing individual items, provide a TOTAL nutrition summary.
4. Explain choices briefly, without comparing to foods not selected.
5. Do NOT mention sources, databases, or internal mechanisms.

If the QUESTION asks for a meal or diet plan:
- Organize the response by meal (e.g., Dinner, Breakfast)
- Choose foods only from the list below
- Give options for breakfast, lunch, dinner and snack. 
- If suitable items are not available for a specific meal, state this explicitly instead of omitting the meal.

QUESTION:
{question}

AVAILABLE FOODS:
{context}
""".strip()


entry_template = """
food_name: {food_name}
meal_type: {meal_type}
food_type: {food_type}
serving_size: {serving_size}
calories_kcal: {calories_kcal}
protein_g: {protein_g}
carbs_g: {carbs_g}
fats_g: {fats_g}
fiber_g: {fiber_g}
saturated_fat_g: {saturated_fat_g}
monounsaturated_fat_g: {monounsaturated_fat_g}
polyunsaturated_fat_g: {polyunsaturated_fat_g}
trans_fat_g: {trans_fat_g}
sodium_mg: {sodium_mg}
calcium_mg: {calcium_mg}
phosphorus_mg: {phosphorus_mg}
""".strip()



def build_prompt(query, search_results):
    context = ""

    for point in search_results:
        context += entry_template.format(**point.payload) + "\n\n"

    prompt = prompt_template.format(
        question=query,
        context=context
    ).strip()

    return prompt



In [31]:
def llm(prompt):

    response = model.generate_content(prompt)
    return response.text

In [32]:
def rag(query, limit=5):
    docs = search(query, limit)
    prompt = build_prompt(query,docs)
    answer = llm(prompt)
    return answer

In [33]:
query = 'Give me a high protein vegetarian diet'

In [34]:
#Test
print(rag(query))

Here is a high-protein vegetarian diet plan based on the available food items. This plan prioritizes foods explicitly listed as vegetarian and high in protein.

### Diet Plan

**Breakfast**
For breakfast, I have selected the Protein Shake and Protein Oats, both of which are excellent vegetarian sources of protein to start your day.

*   **Protein Shake**
    *   Serving size: 100g cooked
    *   Calories: 338 kcal
    *   Protein: 21.19g
    *   Carbohydrates: 29.77g
    *   Fats: 1.45g
*   **Protein Oats**
    *   Serving size: 100g cooked
    *   Calories: 239 kcal
    *   Protein: 19.26g
    *   Carbohydrates: 24.79g
    *   Fats: 8.8g

**Lunch**
No suitable items are available for lunch from the provided food list.

**Dinner**
No suitable items are available for dinner from the provided food list.

**Snack**
For your snack, a combination of Mixed Nuts, Raisins, and Green Salad provides a substantial boost of protein and other essential nutrients.

*   **Mixed Nuts**
    *   Serving

## Retrieval Evaluation

In [37]:
evaluate_query = pd.read_csv('data/nutrirag_ground_truth.csv')

In [38]:
evaluate_query.head()

,query,relevant_foods
0,Build me a vegetarian diet plan,Veg Sandwich|Sprouts Salad|Vegetable Soup|Dal ...
1,Build me a non vegetarian diet plan,Scrambled Eggs|Fish Curry|Egg Curry|Chicken Soup
2,High protein vegetarian dinner,Urad Dal|Besan Chilla|Paneer Curry
3,High protein non veg dinner,Chicken Clear Soup|Egg Curry|Grilled Chicken B...
4,Low calorie vegetarian lunch,Veg Clear Soup|Sprouts Salad


In [40]:
ground_truth = evaluate_query.to_dict(orient = 'records')

In [41]:
ground_truth

[{'query': 'Build me a vegetarian diet plan',
  'relevant_foods': 'Veg Sandwich|Sprouts Salad|Vegetable Soup|Dal Tadka'},
 {'query': 'Build me a non vegetarian diet plan',
  'relevant_foods': 'Scrambled Eggs|Fish Curry|Egg Curry|Chicken Soup'},
 {'query': 'High protein vegetarian dinner',
  'relevant_foods': 'Urad Dal|Besan Chilla|Paneer Curry'},
 {'query': 'High protein non veg dinner',
  'relevant_foods': 'Chicken Clear Soup|Egg Curry|Grilled Chicken Breast'},
 {'query': 'Low calorie vegetarian lunch',
  'relevant_foods': 'Veg Clear Soup|Sprouts Salad'},
 {'query': 'High carb breakfast', 'relevant_foods': 'Oats Porridge|Poha|Upma'},
 {'query': 'High protein snacks',
  'relevant_foods': 'Protein Bar|Mixed Nuts|Sprouts Salad'},
 {'query': 'Low calorie snacks',
  'relevant_foods': 'Fruit Bowl|Veg Clear Soup'},
 {'query': 'Vegetarian snack options',
  'relevant_foods': 'Veg Sandwich|Sprouts Salad'},
 {'query': 'Non veg breakfast',
  'relevant_foods': 'Scrambled Eggs|Egg Sandwich'},
 {'qu

In [47]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [48]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in ground_truth:
        query = q['query']
        relevant_foods = set(q['relevant_foods'].split('|'))
        results = search_function(query)
        relevance = [
            (point.payload['food_name'] in relevant_foods)
            for point in results
        ]

        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }


In [51]:
evaluate(ground_truth, search)

{'hit_rate': 0.9333333333333333, 'mrr': 0.9741269841269841}